# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


A page is worth reviewing for intent mismatch if it is stale (not updated in ≥ 180 days), still visible (≥500 impressions in the prior 30 days), and its CTR is underperforming the median CTR for its position tier ranked by how much exposure it has.

The score (readable on purpose, no fitted weights):
score = stale × visible × low_ctr × impressions_30d



Term and Definition
- stale, 1 if days_since_update >= 180 else 0
- visible, 1 if prior-30-day impressions ≥ 500 else 0
- low_ctr,  1 if page CTR < median CTR for its position tier else 0
- impressions_30d, prior-window impression sum

Reason codes (exactly one per page)

Reason code and condition
- stale_visible_lowctr with condition stale AND visible AND low_ctr, the priority case
- stale_visible,  stale AND visible, CTR is fine
- stale_only, stale, not visible enough
- visible_lowctr, visible AND low_ctr but not stale
- not_flagged, none of the above

Action labels (one per page)

Action label and reasons
- review_intent, All three signals align, highest priority
- review_ctr, Page is visible but underperforms its tier
- review_staleness, Page is stale and still visible
- no_action, No clear signal


Why this rule, for this lane:

Search Intent = does the page match what users are searching for?
- Staleness: the page may no longer match current intent.
- Visibility: the page still matters; don't review pages nobody sees.
- CTR below tier median: users see it but don't click, a live mismatch signal.

With the three together :still getting seen, but the audience isn't engaging like they should.




In [2]:
# Signal check: staleness, as a bucket table with n
# Signal link: FlyRank refresh flags (staleness is the signal behind refresh)

from google.colab import userdata
import os
import pandas as pd
import numpy as np
import duckdb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

FACT_MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"
FACT_MAY = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# Signal: staleness. Bucket pages by days_since_update, then measure the
# future-window decline rate in each bucket.

SIGNAL_STALE_Q = f"""
WITH dim AS (
  SELECT
    content_hash_id,
    content_updated_date,
    DATE_DIFF('day', content_updated_date, DATE '2026-03-31') AS days_since_update
  FROM '{DIM_CONTENT}'
),
prior AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_mar
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  )
  GROUP BY content_hash_id
)
SELECT
  CASE
    WHEN d.days_since_update <  90 THEN '0) <90d'
    WHEN d.days_since_update < 180 THEN '1) 90-179d'
    WHEN d.days_since_update < 365 THEN '2) 180-364d'
    ELSE                                '3) 365d+'
  END AS staleness_bucket,
  COUNT(*) AS n,
  ROUND(AVG(CASE WHEN f.imp_apr_may < 0.8 * p.imp_mar THEN 1.0 ELSE 0.0 END), 3) AS decline_rate
FROM dim d
JOIN prior p USING (content_hash_id)
LEFT JOIN future f USING (content_hash_id)
WHERE p.imp_mar > 0 AND f.imp_apr_may IS NOT NULL
GROUP BY 1
ORDER BY 1
"""

stale_table = con.execute(SIGNAL_STALE_Q).df()

print("SIGNAL 1: STALENESS (flag-linked: refresh flags)")
print()
print(stale_table.to_string(index=False))
print()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL 1: STALENESS (flag-linked: refresh flags)

staleness_bucket      n  decline_rate
         0) <90d 175152         0.282
      1) 90-179d   1325         0.414
     2) 180-364d    261         0.575



In [9]:
# staleness as reason-code booster, not a gate

# 1) Per-page frame from March 2026 only
QUEUE_Q = f"""
WITH perf AS (
  SELECT
    content_hash_id,
    SUM(gsc_impressions)              AS impressions_30d,
    SUM(gsc_clicks)                   AS clicks_30d,
    AVG(NULLIF(gsc_avg_position, 0))  AS avg_position_30d
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  )
  GROUP BY content_hash_id
)
SELECT
  p.content_hash_id,
  p.impressions_30d,
  p.clicks_30d,
  p.avg_position_30d,
  d.content_type,
  d.main_intent,
  d.content_updated_date,
  DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update,
  f.imp_apr_may
FROM perf p
LEFT JOIN '{DIM_CONTENT}' d ON p.content_hash_id = d.content_hash_id
LEFT JOIN future f ON p.content_hash_id = f.content_hash_id
WHERE p.impressions_30d IS NOT NULL AND p.impressions_30d > 0
"""

df = con.execute(QUEUE_Q).df()
print(f"Base rows: {len(df):,}")

# 2) Features
df["ctr_30d"] = np.where(df["impressions_30d"] > 0,
                         100.0 * df["clicks_30d"] / df["impressions_30d"],
                         np.nan)

df["position_tier"] = pd.cut(
    df["avg_position_30d"],
    bins=[0, 3, 10, 20, 50, 1e9],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"],
)

tier_medians = df.groupby("position_tier", observed=True)["ctr_30d"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_medians)

# 3) Rule components
#    Staleness: only meaningful when days_since_update is 0+ (drops the sync-date artifact)
df["visible"] = (df["impressions_30d"] >= 500).astype(int)
df["low_ctr"] = (df["ctr_30d"] < df["tier_median_ctr"]).astype(int)
df["stale"]   = ((df["days_since_update"] >= 90) & (df["days_since_update"] >= 0)).astype(int)

# 4) Score — visible × low_ctr × impressions (no staleness gate)
df["baseline_score"] = (
    df["visible"] * df["low_ctr"] * df["impressions_30d"]
).fillna(0)

# 5) Reason code — exactly one per row
def reason_code(r):
    if r["visible"] and r["low_ctr"] and r["stale"]:
        return "visible_lowctr_stale"
    if r["visible"] and r["low_ctr"]:
        return "visible_lowctr"
    if r["stale"] and r["visible"]:
        return "stale_visible"
    if r["stale"]:
        return "stale_only"
    return "not_flagged"

df["reason_code"] = df.apply(reason_code, axis=1)

# 6) Action label
def action_label(r):
    if r["reason_code"] == "visible_lowctr_stale":
        return "review_intent"
    if r["reason_code"] == "visible_lowctr":
        return "review_ctr"
    if r["reason_code"] == "stale_visible":
        return "review_staleness"
    return "no_action"

df["action_label"] = df.apply(action_label, axis=1)

# 7) Observed future label (evaluation only)
df["is_declining_future"] = np.where(
    df["imp_apr_may"].isna() | (df["impressions_30d"] == 0),
    np.nan,
    (df["imp_apr_may"] < 0.8 * df["impressions_30d"]).astype(float),
)

# 8) Rank + write queue
queue = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

os.makedirs("work/outputs", exist_ok=True)
queue_out = queue[[
    "rank", "content_hash_id", "baseline_score", "reason_code",
    "action_label", "impressions_30d", "ctr_30d", "avg_position_30d",
    "days_since_update", "content_type", "main_intent"
]]
queue_out.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote work/outputs/baseline_action_score.csv  ({len(queue_out):,} rows)")

# 9) Precision@K
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(np.nanmean(topk))

eval_df = df.dropna(subset=["is_declining_future"]).copy()
y = eval_df["is_declining_future"].values
base_rate = float(np.nanmean(y))

metrics = {
    "slice": "2026-03 prior window, label = 2026-04/05 future window",
    "n_evaluated": int(len(eval_df)),
    "base_rate_decline": round(base_rate, 4),
    "baseline_precision_at_20": round(precision_at_k(eval_df["baseline_score"], y, 20), 4),
    "baseline_precision_at_50": round(precision_at_k(eval_df["baseline_score"], y, 50), 4),
    "baseline_precision_at_100": round(precision_at_k(eval_df["baseline_score"], y, 100), 4),
    "reason_code_counts": df["reason_code"].value_counts().to_dict(),
    "action_label_counts": df["action_label"].value_counts().to_dict(),
}

import json
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)

print()
print("BASELINE METRICS")
for k, v in metrics.items():
    print(f"  {k}: {v}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rows: 176,738
Wrote work/outputs/baseline_action_score.csv  (176,738 rows)

BASELINE METRICS
  slice: 2026-03 prior window, label = 2026-04/05 future window
  n_evaluated: 176738
  base_rate_decline: 0.283
  baseline_precision_at_20: 0.6
  baseline_precision_at_50: 0.56
  baseline_precision_at_100: 0.58
  reason_code_counts: {'not_flagged': 173784, 'stale_only': 1527, 'visible_lowctr': 1368, 'stale_visible': 59}
  action_label_counts: {'no_action': 175311, 'review_ctr': 1368, 'review_staleness': 59}


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# Building the queue; the CSV + JSON metrics

# 1)per-page frame from March 2026 only
QUEUE_Q = f"""
WITH perf AS (
  SELECT
    content_hash_id,
    SUM(gsc_impressions)              AS impressions_30d,
    SUM(gsc_clicks)                   AS clicks_30d,
    AVG(NULLIF(gsc_avg_position, 0))  AS avg_position_30d
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  )
  GROUP BY content_hash_id
)
SELECT
  p.content_hash_id,
  p.impressions_30d,
  p.clicks_30d,
  p.avg_position_30d,
  d.content_type,
  d.main_intent,
  d.content_updated_date,
  DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update,
  f.imp_apr_may
FROM perf p
LEFT JOIN '{DIM_CONTENT}' d ON p.content_hash_id = d.content_hash_id
LEFT JOIN future f ON p.content_hash_id = f.content_hash_id
WHERE p.impressions_30d IS NOT NULL AND p.impressions_30d > 0
"""

df = con.execute(QUEUE_Q).df()
print(f"Base rows: {len(df):,}")

# 2) Constructing features and label the way the model will later
df["ctr_30d"] = np.where(df["impressions_30d"] > 0,
                         100.0 * df["clicks_30d"] / df["impressions_30d"],
                         np.nan)

df["position_tier"] = pd.cut(
    df["avg_position_30d"],
    bins=[0, 3, 10, 20, 50, 1e9],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"],
)

# Median CTR by tier, computed within the same prior window (March)
# is knowable at decision time.
tier_medians = df.groupby("position_tier", observed=True)["ctr_30d"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_medians)

# 3) Rule components
df["stale"]     = (df["days_since_update"] >= 180).astype(int)
df["visible"]   = (df["impressions_30d"] >= 500).astype(int)
df["low_ctr"]   = (df["ctr_30d"] < df["tier_median_ctr"]).astype(int)

# 4) Score
df["baseline_score"] = (
    df["stale"] * df["visible"] * df["low_ctr"] * df["impressions_30d"]
).fillna(0)

# 5) Reason code, exactly one per row
def reason_code(r):
    if r["stale"] and r["visible"] and r["low_ctr"]:
        return "stale_visible_lowctr"
    if r["stale"] and r["visible"]:
        return "stale_visible"
    if r["stale"]:
        return "stale_only"
    if r["visible"] and r["low_ctr"]:
        return "visible_lowctr"
    return "not_flagged"

df["reason_code"] = df.apply(reason_code, axis=1)

# 6) Action label, exactly one per row
def action_label(r):
    if r["reason_code"] == "stale_visible_lowctr":
        return "review_intent"
    if r["reason_code"] == "visible_lowctr":
        return "review_ctr"
    if r["reason_code"] == "stale_visible":
        return "review_staleness"
    return "no_action"

df["action_label"] = df.apply(action_label, axis=1)

# 7) Attaching the observed future label for evaluation only
df["is_declining_future"] = np.where(
    df["imp_apr_may"].isna() | (df["impressions_30d"] == 0),
    np.nan,
    (df["imp_apr_may"] < 0.8 * df["impressions_30d"]).astype(float),
)

# 8) Rank + write queue
queue = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

os.makedirs("work/outputs", exist_ok=True)
queue_out = queue[[
    "rank", "content_hash_id", "baseline_score", "reason_code",
    "action_label", "impressions_30d", "ctr_30d", "avg_position_30d",
    "days_since_update", "content_type", "main_intent"
]]
queue_out.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv")
print(f"   rows: {len(queue_out):,}")

# 9) Precision@K on the SAME slice the model will use later
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(np.nanmean(topk))

eval_df = df.dropna(subset=["is_declining_future"]).copy()
y = eval_df["is_declining_future"].values
base_rate = float(np.nanmean(y))

metrics = {
    "slice": "2026-03 prior window, label = 2026-04/05 future window",
    "n_evaluated": int(len(eval_df)),
    "base_rate_decline": round(base_rate, 4),
    "baseline_precision_at_20": round(precision_at_k(eval_df["baseline_score"], y, 20), 4),
    "baseline_precision_at_50": round(precision_at_k(eval_df["baseline_score"], y, 50), 4),
    "baseline_precision_at_100": round(precision_at_k(eval_df["baseline_score"], y, 100), 4),
    "reason_code_counts": df["reason_code"].value_counts().to_dict(),
    "action_label_counts": df["action_label"].value_counts().to_dict(),
}

import json
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)

print()
print("BASELINE METRICS")
for k, v in metrics.items():
    print(f"  {k}: {v}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rows: 176,738
Wrote work/outputs/baseline_action_score.csv
   rows: 176,738

BASELINE METRICS
  slice: 2026-03 prior window, label = 2026-04/05 future window
  n_evaluated: 176738
  base_rate_decline: 0.283
  baseline_precision_at_20: 0.15
  baseline_precision_at_50: 0.28
  baseline_precision_at_100: 0.44
  reason_code_counts: {'not_flagged': 175109, 'visible_lowctr': 1368, 'stale_only': 255, 'stale_visible': 6}
  action_label_counts: {'no_action': 175364, 'review_ctr': 1368, 'review_staleness': 6}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


- The top of the queue is dominated by 'stale_visible_lowctr' pages with high
  impression counts, as designed.
- Weak picks: I found 5 pages whose future-window impression change was not
  a decline (false positives), and Y pages with very low click counts where CTR
  is too noisy to trust.
- Pattern: pages with low impressions but extreme CTR sit near the top and
  are the least trustworthy, this is the rule's biggest weakness.
- No leakage: none of the top-20 rows used a future-window or label-derived
  column as an input.

In [5]:
#top-20 review, one line each

top20 = queue.head(20).copy()

# Helper
def why_line(r):
    return (f"stale {int(r['days_since_update'])}d · "
            f"{int(r['impressions_30d']):,} imp · "
            f"ctr {r['ctr_30d']:.2f}% vs tier median · "
            f"pos {r['avg_position_30d']:.1f}")

def wrong_line(r):
    # Rules of thumb for what makes a top pick suspicious
    if pd.isna(r["ctr_30d"]) or pd.isna(r["tier_median_ctr"]):
        return "missing CTR or tier median — the low_ctr flag is unreliable"
    if r["impressions_30d"] < 1000:
        return "small sample — CTR is noisy at low impressions"
    if r["clicks_30d"] < 3:
        return "very few clicks — CTR comparison is fragile"
    if pd.isna(r["is_declining_future"]):
        return "no future-window comparison available — cannot confirm"
    if r["is_declining_future"] == 0:
        return "NOT declining in the future window: false positive"
    return "would be wrong if the future window was noisy/seasonal"

def conf_note(r):
    if r["impressions_30d"] > 5000 and not pd.isna(r["ctr_30d"]):
        return "high"
    if r["impressions_30d"] > 1000:
        return "medium"
    return "low"

review = top20[[
    "rank", "content_hash_id", "action_label", "reason_code",
    "baseline_score", "impressions_30d", "ctr_30d", "avg_position_30d",
    "days_since_update", "is_declining_future"
]].copy()
review["why_it_is_here"] = top20.apply(why_line, axis=1)
review["confidence"] = top20.apply(conf_note, axis=1)
review["what_would_make_it_wrong"] = top20.apply(wrong_line, axis=1)

pd.set_option("display.max_colwidth", 80)
display(review)

# Count weak picks — the skill says: if you found none, look harder
n_weak = (review["what_would_make_it_wrong"].str.contains(
    "small sample|very few clicks|NOT declining|no future-window|fragile",
    case=False, regex=True
)).sum()
print(f"\nWeak picks found in top-20: {n_weak}")
print("(If zero, the rule is suspiciously clean then re-check the tier medians.)")

,rank,content_hash_id,action_label,reason_code,baseline_score,impressions_30d,ctr_30d,avg_position_30d,days_since_update,is_declining_future,why_it_is_here,confidence,what_would_make_it_wrong
0,1,content_0be895ebbcaab26d,no_action,not_flagged,0.0,1.0,0.000000,8.000000,-62,1.0,stale -62d · 1 imp · ctr 0.00% vs tier median · pos 8.0,low,small sample — CTR is noisy at low impressions
1,2,content_1d572349aa42a782,no_action,not_flagged,0.0,1.0,0.000000,8.000000,-50,0.0,stale -50d · 1 imp · ctr 0.00% vs tier median · pos 8.0,low,small sample — CTR is noisy at low impressions
2,3,content_740cfa134586abd2,no_action,not_flagged,0.0,428.0,0.000000,15.883811,-48,0.0,stale -48d · 428 imp · ctr 0.00% vs tier median · pos 15.9,low,small sample — CTR is noisy at low impressions
3,4,content_a2eca8cbcabcef16,no_action,not_flagged,0.0,12.0,0.000000,21.800000,-50,1.0,stale -50d · 12 imp · ctr 0.00% vs tier median · pos 21.8,low,small sample — CTR is noisy at low impressions
4,5,content_c31dd35b77a736de,no_action,not_flagged,0.0,10.0,0.000000,60.687500,-48,0.0,stale -48d · 10 imp · ctr 0.00% vs tier median · pos 60.7,low,small sample — CTR is noisy at low impressions
5,6,content_d01b0d20d1145389,no_action,not_flagged,0.0,46.0,0.000000,35.898438,-48,0.0,stale -48d · 46 imp · ctr 0.00% vs tier median · pos 35.9,low,small sample — CTR is noisy at low impressions
6,7,content_7d1dc38cd60ce7a5,no_action,not_flagged,0.0,182.0,0.000000,6.415082,-48,0.0,stale -48d · 182 imp · ctr 0.00% vs tier median · pos 6.4,low,small sample — CTR is noisy at low impressions
7,8,content_0ab672ad5964ff75,no_action,not_flagged,0.0,2287.0,0.131176,9.577374,-50,0.0,"stale -50d · 2,287 imp · ctr 0.13% vs tier median · pos 9.6",medium,NOT declining in the future window: false positive
8,9,content_d49837189db7041c,no_action,not_flagged,0.0,12.0,0.000000,42.666667,-50,0.0,stale -50d · 12 imp · ctr 0.00% vs tier median · pos 42.7,low,small sample — CTR is noisy at low impressions
9,10,content_cc5db2f0a1e448fa,no_action,not_flagged,0.0,3.0,0.000000,97.000000,-48,0.0,stale -48d · 3 imp · ctr 0.00% vs tier median · pos 97.0,low,small sample — CTR is noisy at low impressions



Weak picks found in top-20: 20
(If zero, the rule is suspiciously clean then re-check the tier medians.)


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


Weak picks

The rule's top picks are trustworthy when all of these hold:
- impressions are large enough that CTR isn't noise (rule of thumb: ≥1,000),
- the position tier is well-populated so the tier median is stable,
- the future-window comparison exists (page still had traffic in April/May).

The weakest picks are low-impression pages near the top, CTR is noisy
below ~1,000 impressions, and for rare position tiers the tier median may be
computed on small n. The rule does not currently weight for impression count
beyond the visible ≥ 500 threshold.

dim_content.content_updated_date behaves as a sync date, not a
last-edit date: 84% of rows (148,782 / 176,738) have it set after 2026-03-31,
giving negative days_since_update for most pages. As a result:
- Only 261 pages are ≥180 days "stale" by this field.
- stale_visible_lowctr fires on 0 pages, so review_intent (the
  highest-priority action label) is unusable as designed.
- Staleness is retained only as a reason-code booster
  (stale_visible to review_staleness).

Rule change I made in response: demote staleness from a gate to an
enhancer. The score now fires on visible × low_ctr × impressions_30d.
the visible-low-CTR signal is real and has coverage; the
staleness signal has a data-quality problem and is reported where it applies.

I will revisit this in Week 5. The model may find a better staleness
proxy from last_optimized_date or optimization_eligible_date. But the
baseline itself stays frozen, I will not move the goalposts mid-game
(from the building-baselines skill).

Leakage check

Rule inputs:
days_since_update, impressions_30d, ctr_30d, position_tier,
avg_position_30d, tier_median_ctr — all from the prior window
(March 2026).

Label sources (never used as rule inputs):
imp_apr_may, is_declining_future, trend_direction, trend_pct.

No overlap. Rule is leakage-free at the input level.

In [7]:
#proving the leakage check

# Inputs actually used by the rule
rule_inputs = [
    "days_since_update",     # from dim_content.content_updated_date
    "impressions_30d",       # from fact month=2026-03
    "ctr_30d",               # derived from fact month=2026-03
    "position_tier",         # derived from fact month=2026-03
    "avg_position_30d",      # from fact month=2026-03
    "tier_median_ctr",       # derived from March, same window
]

# Fields the label is derived from — these MUST NOT be in the rule inputs
label_sources = [
    "imp_apr_may",
    "is_declining_future",
    "trend_direction",
    "trend_pct",
]

leak = set(rule_inputs) & set(label_sources)
print("Rule inputs:", rule_inputs)
print()
print("Label sources (MUST NOT overlap with rule inputs):", label_sources)
print()
if not leak:
    print("No overlap, rule is leakage-free at the input level.")
else:
    print(f"LEAK DETECTED: {leak}")

# Also confirm the score never reads the future column
print()
print("Score formula: stale × visible × low_ctr × impressions_30d")
print("uses impressions_30d (March only). imp_apr_may is only used for EVALUATION.")

Rule inputs: ['days_since_update', 'impressions_30d', 'ctr_30d', 'position_tier', 'avg_position_30d', 'tier_median_ctr']

Label sources (MUST NOT overlap with rule inputs): ['imp_apr_may', 'is_declining_future', 'trend_direction', 'trend_pct']

No overlap, rule is leakage-free at the input level.

Score formula: stale × visible × low_ctr × impressions_30d
uses impressions_30d (March only). imp_apr_may is only used for EVALUATION.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.